# Session 6 — Inference, Deployment & Wrap-Up

**Duration:** 75 min  
**Repository:** [MiniMind-Colab](https://github.com/your-org/minimind-colab)

| Part | Topic | Time |
|------|-------|------|
| A | Autoregressive Generation | 20 min |
| B | Interactive Chat | 15 min |
| C | OpenAI-Compatible API | 25 min |
| D | Course Wrap-Up | 15 min |
| — | Wrap-up & Q\&A | 5 min |

In [ ]:
# Cell — Environment setup
!pip install torch transformers datasets fastapi pydantic --quiet

import math, json, os, time, random, re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from contextlib import nullcontext

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}  CUDA available: {torch.cuda.is_available()}')
print(f'Device: {device}')

## Part A — Autoregressive Generation (20 min)

Autoregressive generation produces text **one token at a time**. At each step
the model outputs logits over the full vocabulary, and we apply a chain of
sampling strategies:

1. **Temperature scaling** — divide logits by `T` to sharpen (T < 1) or
   flatten (T > 1) the distribution.
2. **Repetition penalty** — reduce logits for tokens that already appear in
   the sequence, discouraging repetitive text.
3. **Top-k filtering** — keep only the `k` highest-scoring tokens, setting
   everything else to −∞.
4. **Top-p (nucleus) filtering** — keep the smallest set of tokens whose
   cumulative probability exceeds `p`.

In [ ]:
# Cell — Sampling helpers scaffold

torch.manual_seed(42)

# Synthetic logits to test with (batch=1, vocab=100)
vocab_size = 100
raw_logits = torch.randn(1, vocab_size, device=device)
# Fake "previously generated" token ids for repetition penalty testing
prev_tokens = torch.tensor([[2, 5, 10, 15, 20, 5, 10]], device=device)

print(f'Raw logits shape: {raw_logits.shape}')
print(f'Previous tokens shape: {prev_tokens.shape}')

In [ ]:
# Cell — Temperature scaling + repetition penalty

def apply_temperature_and_penalty(logits, input_ids, temperature=0.85, repetition_penalty=1.0):
    """Scale logits by temperature and penalise repeated tokens."""
    # ============================================================
    # TODO: Temperature scaling + repetition penalty
    #
    # 1. Divide logits by `temperature` (element-wise).
    # 2. If repetition_penalty != 1.0, loop over each row in the
    #    batch.  For each row, find the unique token ids that
    #    already appear in `input_ids[i]` and divide those logits
    #    by `repetition_penalty`.
    #
    # Input:  logits      — (batch, vocab) float tensor
    #         input_ids   — (batch, seq_len) long tensor
    #         temperature — float scalar > 0
    #         repetition_penalty — float scalar >= 1.0
    # Output: logits      — (batch, vocab) modified in-place
    #
    # Hint: torch.unique, in-place /= operator
    # ============================================================
    raise NotImplementedError("Fill in the TODO above")

print('apply_temperature_and_penalty defined ✓')

In [ ]:
# Cell — Top-k filtering

def apply_top_k(logits, top_k=50):
    """Zero out all logits below the k-th largest value."""
    # ============================================================
    # TODO: Top-k filtering
    #
    # 1. Use torch.topk to find the k largest values in each row.
    # 2. The k-th largest value is at index [..., -1, None].
    # 3. Set all logits below that threshold to -float('inf').
    #
    # Input:  logits — (batch, vocab) float tensor
    #         top_k  — int, number of top values to keep
    # Output: logits — (batch, vocab) with sub-top-k entries = -inf
    #
    # Hint: torch.topk(logits, top_k)[0][..., -1, None] gives the
    #       threshold; use boolean masking.
    # ============================================================
    raise NotImplementedError("Fill in the TODO above")

print('apply_top_k defined ✓')

In [ ]:
# Cell — Top-p (nucleus) filtering

def apply_top_p(logits, top_p=0.85):
    """Keep the smallest set of tokens whose cumulative probability > p."""
    # ============================================================
    # TODO: Top-p (nucleus) filtering
    #
    # 1. Sort logits descending: sorted_logits, sorted_indices.
    # 2. Compute softmax of sorted_logits, then cumulative sum.
    # 3. Build a boolean mask where cumsum > top_p.
    # 4. Shift mask right by one so the first token above the
    #    threshold is kept: mask[..., 1:] = mask[..., :-1].clone(),
    #    mask[..., 0] = 0.
    # 5. Scatter the mask back to original order and set masked
    #    positions to -inf.
    #
    # Input:  logits — (batch, vocab) float tensor
    #         top_p  — float in (0, 1]
    # Output: logits — (batch, vocab) with low-prob entries = -inf
    #
    # Hint: torch.sort, torch.cumsum, torch.softmax, mask.scatter
    # ============================================================
    raise NotImplementedError("Fill in the TODO above")

print('apply_top_p defined ✓')

### ✅ Milestone 1: Sampling Produces Valid Text

We verify that temperature scaling, top-k, and top-p filtering work together
to produce a valid probability distribution (no NaN, correct number of
non-inf entries).

In [ ]:
# Cell — ✅ Milestone 1: Sampling produces valid text

torch.manual_seed(42)
test_logits = torch.randn(1, vocab_size, device=device)

# Temperature + repetition penalty
scaled = apply_temperature_and_penalty(test_logits.clone(), prev_tokens, temperature=0.85, repetition_penalty=1.2)
assert scaled is not None, 'apply_temperature_and_penalty returned None'
assert scaled.shape == (1, vocab_size), f'Wrong shape: {scaled.shape}'
print(f'After temperature+penalty — shape: {scaled.shape}, max: {scaled.max().item():.4f}')

# Top-k filtering
topk_out = apply_top_k(scaled.clone(), top_k=10)
non_inf_count = (topk_out > -float('inf')).sum().item()
assert non_inf_count == 10, f'Expected 10 non-inf values after top-k, got {non_inf_count}'
print(f'After top-k=10 — non-inf entries: {non_inf_count}')

# Top-p filtering
topp_out = apply_top_p(test_logits.clone() / 0.85, top_p=0.85)
non_inf_p = (topp_out > -float('inf')).sum().item()
assert non_inf_p > 0, 'Top-p removed all tokens!'
assert non_inf_p <= vocab_size, f'Top-p result has more entries than vocab'
probs = torch.softmax(topp_out, dim=-1)
assert not torch.isnan(probs).any(), 'NaN in probability distribution after top-p'
print(f'After top-p=0.85 — non-inf entries: {non_inf_p}')

# Combined pipeline: generate one token
combined = apply_temperature_and_penalty(torch.randn(1, vocab_size, device=device), prev_tokens)
combined = apply_top_k(combined, top_k=50)
combined = apply_top_p(combined, top_p=0.85)
token = torch.multinomial(torch.softmax(combined, dim=-1), num_samples=1)
assert token.shape == (1, 1), f'Sampled token wrong shape: {token.shape}'
assert 0 <= token.item() < vocab_size, f'Token id out of range: {token.item()}'
print(f'Sampled token id: {token.item()}')

print('✅ Milestone 1 passed — sampling pipeline produces valid output')

### ✅ Milestone 2: Top-k Filtered Logits

We verify that after top-k filtering, exactly `k` values remain finite
and the rest are negative infinity.

In [ ]:
# Cell — ✅ Milestone 2: Top-k filtered logits

torch.manual_seed(99)
for k in [5, 10, 25]:
    test_l = torch.randn(2, vocab_size, device=device)
    filtered = apply_top_k(test_l.clone(), top_k=k)
    for b in range(2):
        n_valid = (filtered[b] > -float('inf')).sum().item()
        assert n_valid == k, f'Batch {b}, top_k={k}: expected {k} non-inf, got {n_valid}'
    print(f'top_k={k:2d} — each row has exactly {k} finite values ✓')

print('✅ Milestone 2 passed — top-k filtering is correct')

## Part B — Interactive Chat (15 min)

The `eval_llm.py` script in the repository provides a simple interactive
chat loop.  Here we reproduce the essential logic: loading the model +
tokenizer, encoding a prompt, calling `model.generate()`, and decoding the
result.

### Key concepts

- **Chat template** — wraps the user message in special tokens so the model
  recognises the conversational structure.
- **`model.generate()`** — orchestrates the full autoregressive loop
  (temperature, top-k, top-p, repetition penalty, EOS stopping).
- **Streamer** — optionally prints tokens as they are generated.

In [ ]:
# Cell — Chat template helper

def format_chat_prompt(messages, system_prompt='You are a helpful assistant.'):
    """Format a list of {role, content} dicts into a single prompt string.

    Uses the ChatML-style template:
        <s>system\n{system}\n</s>
        <s>user\n{user}\n</s>
        <s>assistant\n
    """
    parts = [f'<s>system\n{system_prompt}\n</s>']
    for msg in messages:
        role = msg['role']
        content = msg['content']
        if role == 'user':
            parts.append(f'<s>user\n{content}\n</s>')
        elif role == 'assistant':
            parts.append(f'<s>assistant\n{content}\n</s>')
    # Open the assistant turn for generation
    parts.append('<s>assistant\n')
    return '\n'.join(parts)

# Quick demo
demo_msgs = [{'role': 'user', 'content': 'What is 2+2?'}]
print(format_chat_prompt(demo_msgs))
print('format_chat_prompt defined ✓')

In [ ]:
# Cell — Simulated generation demo

# Since we may not have the full MiniMind weights available in this notebook
# environment, we simulate the generation loop to illustrate the pipeline.

def simulate_generate(prompt_ids, vocab_size=100, max_new_tokens=20,
                      temperature=0.85, top_k=50, top_p=0.85,
                      repetition_penalty=1.0, eos_token_id=2):
    """Simulate autoregressive generation using our sampling functions."""
    input_ids = prompt_ids.clone()
    for step in range(max_new_tokens):
        # In a real model, logits come from a forward pass;
        # here we use random logits for demonstration.
        logits = torch.randn(1, vocab_size, device=device)

        # Apply our sampling pipeline
        logits = apply_temperature_and_penalty(
            logits, input_ids, temperature=temperature,
            repetition_penalty=repetition_penalty
        )
        logits = apply_top_k(logits, top_k=top_k)
        logits = apply_top_p(logits, top_p=top_p)

        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_token], dim=-1)

        if next_token.item() == eos_token_id:
            break

    return input_ids

# Run simulation
torch.manual_seed(42)
prompt = torch.tensor([[1, 50, 30, 22]], device=device)
output = simulate_generate(prompt, max_new_tokens=20, eos_token_id=2)
generated = output[0, prompt.shape[1]:]
print(f'Prompt length: {prompt.shape[1]}')
print(f'Generated {generated.shape[0]} new tokens: {generated.tolist()}')
assert generated.shape[0] > 0, 'Generation produced no tokens!'
print('simulate_generate works ✓')

## Part C — OpenAI-Compatible API (25 min)

To make the model accessible to any client that speaks the OpenAI chat
protocol, we wrap it in a **FastAPI** server.  The key components are:

1. **`ChatRequest`** — a Pydantic model that validates incoming JSON.
2. **`/v1/chat/completions`** — the endpoint that handles both streaming
   (SSE) and non-streaming responses.
3. **`parse_response()`** — post-processes the raw model output to extract
   `<think>` reasoning blocks and `<tool_call>` structured calls.

In [ ]:
# Cell — FastAPI scaffold

from pydantic import BaseModel

# We define the API types here; no server is started in the notebook.

class ChatRequest(BaseModel):
    model: str = 'minimind'
    messages: list = []
    temperature: float = 0.7
    top_p: float = 0.92
    max_tokens: int = 8192
    stream: bool = True

# Verify Pydantic model
req = ChatRequest(messages=[{'role': 'user', 'content': 'Hello'}])
print(f'ChatRequest fields: {list(req.model_fields.keys())}')
print(f'Default temperature: {req.temperature}')
print(f'Default stream: {req.stream}')
print('ChatRequest defined ✓')

In [ ]:
# Cell — generate_stream_response scaffold

def generate_stream_response(messages, temperature=0.7, top_p=0.92, max_tokens=100):
    """Yield chunks of generated text (simulated for notebook demo).

    In production (serve_openai_api.py) this calls model.generate with a
    TextIteratorStreamer and yields SSE-formatted chunks.
    """
    # Simulated response for demonstration
    response_text = 'This is a simulated response from MiniMind.'
    words = response_text.split()
    for i, word in enumerate(words):
        chunk = {
            'id': 'chatcmpl-demo',
            'object': 'chat.completion.chunk',
            'choices': [{
                'index': 0,
                'delta': {'content': word + ' '},
                'finish_reason': None if i < len(words) - 1 else 'stop'
            }]
        }
        yield json.dumps(chunk)
    yield '[DONE]'

# Test streaming
chunks = list(generate_stream_response([{'role': 'user', 'content': 'Hi'}]))
print(f'Stream produced {len(chunks)} chunks')
assert len(chunks) > 1, 'Stream should produce multiple chunks'
assert chunks[-1] == '[DONE]', 'Stream should end with [DONE]'
print('generate_stream_response defined ✓')

In [ ]:
# Cell — Endpoint logic walkthrough

def chat_completions_handler(request):
    """Simulates the /v1/chat/completions endpoint logic."""
    if request.stream:
        # In production: return StreamingResponse(...)
        chunks = list(generate_stream_response(
            messages=request.messages,
            temperature=request.temperature,
            top_p=request.top_p,
            max_tokens=request.max_tokens
        ))
        return {'type': 'stream', 'chunks': chunks}
    else:
        # Non-streaming: single response
        answer = 'This is a non-streaming response.'
        return {
            'choices': [{
                'message': {'role': 'assistant', 'content': answer}
            }]
        }

# Test non-streaming
req_sync = ChatRequest(messages=[{'role': 'user', 'content': 'Hello'}], stream=False)
result = chat_completions_handler(req_sync)
assert 'choices' in result, 'Non-streaming response missing choices'
print(f'Non-streaming response: {result["choices"][0]["message"]["content"][:50]}')

# Test streaming
req_stream = ChatRequest(messages=[{'role': 'user', 'content': 'Hi'}], stream=True)
result_s = chat_completions_handler(req_stream)
assert result_s['type'] == 'stream', 'Should be a stream response'
print(f'Streaming response chunks: {len(result_s["chunks"])}')

print('chat_completions_handler works ✓')

In [ ]:
# Cell — parse_response implementation

def parse_response(text):
    """Extract <think> reasoning and <tool_call> blocks from raw model output.

    Returns (cleaned_text, reasoning_content, tool_calls_or_None).
    """
    # ============================================================
    # TODO: parse_response — extract think blocks and tool calls
    #
    # 1. Search for <think>...</think> using re.search with
    #    re.DOTALL.  If found, capture group(1).strip() as
    #    reasoning_content.  Remove the matched region from text
    #    using re.sub.
    # 2. Find all <tool_call>...</tool_call> blocks using
    #    re.findall with re.DOTALL.
    # 3. For each match, strip whitespace and json.loads it.
    #    Build a list of dicts:
    #      {"type": "function",
    #       "function": {"name": call["name"],
    #                    "arguments": json.dumps(call.get("arguments", {}))}}
    #    Skip any match that fails to parse.
    # 4. Return (text.strip(), reasoning_content, tool_calls or None).
    #    — tool_calls should be None if the list is empty.
    #
    # Input:  text — str, raw model output
    # Output: (str, str|None, list|None)
    #
    # Hint: re.search, re.findall, re.sub, json.loads, json.dumps
    # ============================================================
    raise NotImplementedError("Fill in the TODO above")

print('parse_response defined ✓')

### ✅ Milestone 3: parse_response Extracts Think & Tool Calls

We verify that `parse_response` correctly pulls out `<think>` reasoning
content and `<tool_call>` structured invocations from raw model output.

In [ ]:
# Cell — ✅ Milestone 3: parse_response verification

# Test 1: text with <think> block
think_text = '<think>Let me reason about this carefully.</think>The answer is 42.'
cleaned, reasoning, tools = parse_response(think_text)
print(f'Cleaned text: {cleaned!r}')
print(f'Reasoning: {reasoning!r}')
print(f'Tool calls: {tools}')
assert reasoning == 'Let me reason about this carefully.', f'Wrong reasoning: {reasoning!r}'
assert 'The answer is 42' in cleaned, f'Cleaned text missing answer: {cleaned!r}'
assert tools is None, f'Should have no tool calls, got: {tools}'

# Test 2: text with <tool_call> block
tool_text = 'I will check. <tool_call>{"name": "get_weather", "arguments": {"city": "Tokyo"}}</tool_call> Done.'
cleaned2, reasoning2, tools2 = parse_response(tool_text)
print(f'\nCleaned text: {cleaned2!r}')
print(f'Reasoning: {reasoning2}')
print(f'Tool calls: {tools2}')
assert reasoning2 is None, f'Should have no reasoning, got: {reasoning2!r}'
assert tools2 is not None, 'Should have tool calls'
assert len(tools2) == 1, f'Expected 1 tool call, got {len(tools2)}'
assert tools2[0]['function']['name'] == 'get_weather', f'Wrong tool name: {tools2[0]}'

# Test 3: both <think> and <tool_call>
both_text = '<think>I need weather data.</think>Let me check. <tool_call>{"name": "get_weather", "arguments": {"city": "Paris"}}</tool_call>'
cleaned3, reasoning3, tools3 = parse_response(both_text)
assert reasoning3 == 'I need weather data.', f'Wrong reasoning: {reasoning3!r}'
assert tools3 is not None and len(tools3) == 1, 'Should have 1 tool call'
assert tools3[0]['function']['name'] == 'get_weather'
print(f'\nBoth blocks — reasoning: {reasoning3!r}, tool: {tools3[0]["function"]["name"]}')

# Test 4: plain text (no special blocks)
plain = 'Hello, how can I help you?'
cleaned4, reasoning4, tools4 = parse_response(plain)
assert reasoning4 is None, 'Plain text should have no reasoning'
assert tools4 is None, 'Plain text should have no tool calls'
assert cleaned4 == plain, f'Plain text should be unchanged: {cleaned4!r}'
print(f'Plain text — unchanged ✓')

print('✅ Milestone 3 passed — parse_response correctly extracts thinking content and tool calls')

## Part D — Course Wrap-Up (15 min)

We now have every piece of the LLM pipeline — from raw text to a deployed
chat API.  Let's review the full journey.

### Full 6-Session Pipeline Review

| Session | Title | Key Topics |
|---------|-------|------------|
| **1** | Tokenizer + Architecture | BPE tokenization, RMSNorm, RoPE positional encoding, Grouped-Query Attention (GQA) |
| **2** | Model + Pretraining | SwiGLU FFN, Mixture of Experts (MoE), next-token prediction, cross-entropy loss |
| **3** | SFT + LoRA | Supervised fine-tuning, label masking, LoRA parameter-efficient adaptation |
| **4** | Alignment | DPO preference optimization, GRPO policy-gradient alignment |
| **5** | Agent RL + Distillation | Tool calling with `<tool_call>` XML, reward functions, KL-divergence knowledge distillation |
| **6** | Inference + Deployment | Temperature / top-k / top-p sampling, repetition penalty, FastAPI serving |

### The complete pipeline

```
┌────────────────────────────────────────────────────────────────────┐
│                         MiniMind Pipeline                          │
├────────────┬───────────────────────────────────────────────────────┤
│ Session 1  │  Tokenizer (BPE)  →  Architecture (RMSNorm, RoPE,   │
│            │                       GQA, Transformer blocks)       │
├────────────┼───────────────────────────────────────────────────────┤
│ Session 2  │  Model init  →  Pretraining (next-token prediction,  │
│            │                  SwiGLU FFN, optional MoE routing)   │
├────────────┼───────────────────────────────────────────────────────┤
│ Session 3  │  SFT (label masking, chat format)  →  LoRA           │
│            │  (low-rank adapters, α / r scaling)                  │
├────────────┼───────────────────────────────────────────────────────┤
│ Session 4  │  DPO (preference pairs, implicit reward)  →  GRPO   │
│            │  (group-relative policy gradient)                    │
├────────────┼───────────────────────────────────────────────────────┤
│ Session 5  │  Agent RL (tool-call rewards, multi-turn rollouts)   │
│            │  →  Knowledge Distillation (KL-div, T² scaling)      │
├────────────┼───────────────────────────────────────────────────────┤
│ Session 6  │  Inference (temperature, top-k, top-p, rep-penalty)  │
│            │  →  Deployment (FastAPI, OpenAI-compatible API)       │
└────────────┴───────────────────────────────────────────────────────┘
```

### Key takeaways per session

**Session 1 — Tokenizer + Architecture**
- **BPE** builds a subword vocabulary by iteratively merging the most frequent byte pairs.
- **RMSNorm** stabilises activations with root-mean-square normalisation (no mean subtraction).
- **RoPE** encodes position via rotation matrices applied to Q/K pairs — no learned positional embeddings.
- **GQA** shares key/value heads across multiple query heads, reducing KV-cache memory.

**Session 2 — Model + Pretraining**
- **SwiGLU FFN** uses gated linear units with the SiLU activation for better gradient flow.
- **Mixture of Experts (MoE)** routes each token to a subset of expert FFNs, scaling capacity without proportional compute.
- **Next-token prediction** is the self-supervised objective: cross-entropy between predicted and actual next token.

**Session 3 — SFT + LoRA**
- **Label masking** ignores the prompt tokens in the loss so the model only learns to produce the assistant response.
- **LoRA** injects low-rank matrices ($\Delta W = BA$) into attention projections, keeping most parameters frozen.
- The rank $r$ and scaling factor $\alpha$ control the capacity vs. efficiency trade-off.

**Session 4 — Alignment**
- **DPO** optimises the policy directly from preference pairs — no explicit reward model needed.
- **GRPO** uses group-relative advantages: rewards are normalised within a batch of rollouts.
- Both methods improve helpfulness and safety without catastrophic forgetting of pretrained knowledge.

**Session 5 — Agent RL + Distillation**
- **Tool calling** lets the model emit structured `<tool_call>` blocks that an executor can dispatch.
- **Reward shaping** scores responses on tool-name accuracy, argument correctness, and response length.
- **Knowledge distillation** trains a small student model to match the soft output distribution of a larger teacher.

**Session 6 — Inference + Deployment**
- **Temperature** controls randomness: lower values sharpen the distribution, higher values flatten it.
- **Top-k** restricts sampling to the k most likely tokens, preventing low-probability noise.
- **Top-p (nucleus)** dynamically adapts the candidate set to the shape of the distribution.
- **Repetition penalty** discourages the model from repeating tokens already generated.
- A **FastAPI server** with the `/v1/chat/completions` endpoint makes the model compatible with any OpenAI client.

In [ ]:
# Cell — Pipeline summary visualisation

pipeline_stages = [
    ('Session 1', 'Tokenizer + Architecture', 'BPE, RMSNorm, RoPE, GQA'),
    ('Session 2', 'Model + Pretraining', 'FFN, MoE, next-token prediction'),
    ('Session 3', 'SFT + LoRA', 'Label masking, low-rank adapters'),
    ('Session 4', 'Alignment', 'DPO preference, GRPO policy gradient'),
    ('Session 5', 'Agent RL + Distillation', 'Tool calling, KL-div distillation'),
    ('Session 6', 'Inference + Deployment', 'Sampling strategies, FastAPI API'),
]

print('=' * 65)
print('  MiniMind — Complete Training & Deployment Pipeline')
print('=' * 65)
for session, title, details in pipeline_stages:
    print(f'  {session:12s} │ {title:30s} │ {details}')
print('=' * 65)
print()
print('Congratulations — you have built a complete LLM from scratch! 🎉')

## Wrap-up (5 min)

### What we built today

| Component | Key idea |
|-----------|----------|
| **apply_temperature_and_penalty** | Logit scaling and repeated-token suppression |
| **apply_top_k** | Keep only the k highest-scoring candidate tokens |
| **apply_top_p** | Dynamic nucleus filtering by cumulative probability |
| **parse_response** | Extract `<think>` reasoning and `<tool_call>` structured blocks |

### Training pipeline

```
Pretrain → SFT → LoRA → DPO / GRPO → Agent RL + Distillation → Inference & API
                                                                  ↑ Session 6
```

### Key takeaways
- **Temperature** is the simplest control knob — it trades off quality vs. diversity.
- **Top-k** and **top-p** are complementary: top-k gives a hard cap, top-p adapts to the distribution shape.
- **Repetition penalty** is critical for open-ended generation to avoid degenerate loops.
- An **OpenAI-compatible API** with FastAPI and SSE streaming lets any chat client use your model.
- The complete MiniMind pipeline — from tokenizer to deployed API — fits in six 75-minute sessions.

### 🎓 Congratulations!

You have now built, trained, aligned, and deployed a complete language model
from scratch.  Every layer — BPE tokenization, RMSNorm, RoPE, GQA attention,
SwiGLU FFN, MoE routing, SFT, LoRA, DPO, GRPO, agent RL, distillation,
and inference — was implemented by hand.  Go build something great! 🚀